# Résultats pipeline — CORTI 500

Features absolues · Isolation Forest + Autoencoder + PCA

In [5]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.loader import load_json_file
from src.features import build_feature_matrix, preprocess, VISIT_CATEGORY_LABELS
from src.models.unsupervised import run_unsupervised_pipeline
from src.evaluate import (
    plot_audiogram,
    plot_top_anomalies,
    plot_patient_trajectory,
    summary_report,
    plot_flag_overlap,
    plot_anomaly_score_distribution,
)

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

In [ ]:
records = load_json_file('../CORTI_sample_audiograms_500.json')
df = pd.DataFrame(records)

print(f"Records : {len(df)}")
print(f"Patients uniques : {df['patient'].nunique()}")
print()
visit_counts = df['visit_category'].value_counts().sort_index()
for cat, count in visit_counts.items():
    label = VISIT_CATEGORY_LABELS.get(cat, '?')
    print(f"  {cat} {label:10s} : {count}")

In [ ]:
feature_df, feature_cols = build_feature_matrix(df)
X, scaler, imputer = preprocess(feature_df, fit=True)

print(f"Matrice features : {X.shape}")

scores_df, if_model, ae_model, _ = run_unsupervised_pipeline(
    X,
    contamination=0.05,
    ae_epochs=100,
    feature_df=feature_df,
)

summary_report(df, scores_df)

n_nihl = scores_df["nihl_flag"].sum() if "nihl_flag" in scores_df.columns else "N/A"
print(f"\n{'nihl_flag (creux >15 dB, 2–8 kHz)':35s} : {n_nihl:>4} / {len(df)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
plot_anomaly_score_distribution(scores_df['anomaly_score_if'], title='Isolation Forest', ax=axes[0])
plot_anomaly_score_distribution(scores_df['reconstruction_error'], title='Autoencoder', ax=axes[1])
plot_anomaly_score_distribution(scores_df['pca_reconstruction_error'], title='PCA baseline', ax=axes[2])
plt.tight_layout()
plt.show()

plot_flag_overlap(scores_df)
plt.show()

In [ ]:
GENDER_LABEL = {1: 'Homme', 2: 'Femme'}

flagged_idx = scores_df[scores_df['anomaly_consensus'] == 1].index.tolist()
normal_idx  = scores_df[scores_df['anomaly_consensus'] == 0].index.tolist()
normal_df   = df.iloc[normal_idx]

print(f"{len(flagged_idx)} anomalies  |  {len(normal_idx)} cas normaux\n")

for idx in flagged_idx[:8]:
    row    = df.iloc[idx]
    score  = scores_df.loc[idx, 'reconstruction_error']
    age    = row['age_at_visit']
    gender = GENDER_LABEL.get(row['gender'], '?')
    visit_label = VISIT_CATEGORY_LABELS.get(row['visit_category'], '?')

    # Infos encoche
    depth_L = feature_df.loc[idx, 'notch_depth_L']
    depth_R = feature_df.loc[idx, 'notch_depth_R']
    freq_L  = feature_df.loc[idx, 'notch_freq_L']
    freq_R  = feature_df.loc[idx, 'notch_freq_R']
    notch_str = ""
    if depth_L > 0:
        notch_str += f" | OG: creux {freq_L/1000:.0f}kHz={depth_L:.0f}dB"
    if depth_R > 0:
        notch_str += f" | OD: creux {freq_R/1000:.0f}kHz={depth_R:.0f}dB"

    # Référence normale même sexe, âge proche
    pool = normal_df[normal_df['gender'] == row['gender']]
    if pool.empty:
        pool = normal_df
    if not np.isnan(age):
        ref_row = pool.iloc[(pool['age_at_visit'] - age).abs().argmin()]
    else:
        ref_row = pool.iloc[0]
    ref_score  = scores_df.loc[ref_row.name, 'reconstruction_error']
    ref_age    = ref_row['age_at_visit']
    ref_gender = GENDER_LABEL.get(ref_row['gender'], '?')

    age_str     = f"{age:.0f} ans"     if not np.isnan(age)     else "âge ?"
    ref_age_str = f"{ref_age:.0f} ans" if not np.isnan(ref_age) else "âge ?"

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    plot_audiogram(
        row['dots_left'], row['dots_right'],
        title=f"ANOMALIE — {gender}, {age_str}, {visit_label} | err={score:.3f}{notch_str}",
        ax=axes[0],
    )
    plot_audiogram(
        ref_row['dots_left'], ref_row['dots_right'],
        title=f"RÉFÉRENCE NORMALE — {ref_gender}, {ref_age_str} | err={ref_score:.3f}",
        ax=axes[1],
    )
    plt.tight_layout()
    plt.show()